# Merge CSV results
Loads multiple CSV files, concatenates them with pandas, removes duplicate rows
based on the primary key (Model Name, Dataset Size), and writes `results.csv`.

In [4]:
import pandas as pd
from pathlib import Path

root = Path(".")  # notebook directory (project root)
# Collect CSV files in the current directory (exclude the output file if present)
csv_files = sorted([p for p in root.glob("*.csv") if p.name != "results.csv"])
if not csv_files:
    print("No CSV files found in", root.resolve())
else:
    print("Found CSV files:", [p.name for p in csv_files])

dfs = []
required_cols = {"Model Name", "Dataset Size"}
for p in csv_files:
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f"Skipping {p.name}: read error: {e}")
        continue
    if not required_cols.issubset(df.columns):
        print(
            f"Skipping {p.name}: missing required columns {required_cols - set(df.columns)}"
        )
        continue
    dfs.append(df)

if not dfs:
    raise SystemExit("No valid CSV dataframes to merge.")

merged = pd.concat(dfs, ignore_index=True)
before = len(merged)
merged = merged.drop_duplicates(
    subset=[
        "Model Name",
        "Dataset Size",
        "Translation X",
        " Translation Y",
        "Rotation",
        "Scale",
        "Shear X",
        " Shear Y",
    ],
    keep="first",
)
after = len(merged)
print(f"Rows before dedupe: {before}; after dedupe: {after}")
out = root / "results.csv"
merged.to_csv(out, index=False)
print("Saved merged results to", out.resolve())

Found CSV files: ['results_2026-03-25.csv', 'results_2026-03-26.csv', 'results_2026-03-27_1.csv', 'results_2026-03-27_2.csv', 'results_2026-03-27_3.csv', 'results_2026-03-28.csv']
Rows before dedupe: 349; after dedupe: 256
Saved merged results to C:\Users\ericr\OneDrive\Documentos\Universitat\DTU_ADLCV\ADLCV_Project\results.csv
